In [8]:
import pandas as pd
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import bioframe as bf

import plotly.express as px
import matplotlib.pyplot as plt


from lib.utils import read_parameters, replace_filename

pd.options.display.max_columns = 100

# Functions

In [9]:
def training(df, test_chrom):
    """ Trains a model returns classifications"""

    X_train = df.loc[df['chrom'] != test_chrom, features]
    X_test = df.loc[df['chrom'] == test_chrom, features]
    y_train = df.loc[df['chrom'] != test_chrom, 'confirmed']
    y_test = df.loc[df['chrom'] == test_chrom, 'confirmed']

    RF = RandomForestClassifier(n_estimators=100)
    RF.fit(X_train, y_train)

    predictions = RF.predict(X_test)
    X_test['confirmed'] = y_test
    X_test['predictions'] = predictions
    cm = confusion_matrix(y_test, predictions)
    tn = cm[0][0]
    fn = cm[0][1]
    fp = cm[1][0]
    tp = cm[1][1]

    fp_svs = list(X_test.loc[(X_test['confirmed'] == 0) & (X_test['predictions'] == 1)].index)
    fn_svs = list(X_test.loc[(X_test['confirmed'] == 1) & (X_test['predictions'] == 0)].index)

    return fp_svs, fn_svs

In [10]:
def load_sample_data(SAMPLE, REF):
    """ Load sample data from a single sample. """
    
    FEATURE_DIR = f'/confidential/FamilyR13/DATA/10x/sv_compare/results/{SAMPLE}_{REF}/ensemble'
    df_raw = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.raw.tsv', sep='\t')
    df_ref = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.ref.tsv', sep='\t', low_memory=False)

    filenames_aln_ill = glob(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.aln.ill.*.tsv')
    df_aln_ill = pd.concat([pd.read_csv(f, sep='\t') for f in filenames_aln_ill], ignore_index=True)

    df = df_raw.merge(df_ref.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='left')
    df = df.merge(df_aln_ill.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='inner')

    return df

# Script

In [16]:
# PARAMETERS
SAMPLES = ['17-08']
REF = 'hg38'
TYPE = 'DEL'
CHROMS = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 
          'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY']

In [17]:
# Load Data
dfs = []
for SAMPLE in SAMPLES:
    df = load_sample_data(SAMPLE, REF)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [22]:
# Filter NaNs
features = ['size'] + list(df.columns[13:])
df = df.dropna(subset=features).copy().reset_index(drop=True)

# Select SV Type
df = df[df['type'] == 'DEL'].copy().reset_index(drop=True)

In [25]:
test_chrom = 'chr1'

X_train = df.loc[df['chrom'] != test_chrom, features]
X_test = df.loc[df['chrom'] == test_chrom, features]
y_train = df.loc[df['chrom'] != test_chrom, 'confirmed']
y_test = df.loc[df['chrom'] == test_chrom, 'confirmed']

RF = RandomForestClassifier(n_estimators=100)
RF.fit(X_train, y_train)

predictions = RF.predict(X_test)
X_test['confirmed'] = y_test
X_test['predictions'] = predictions
cm = confusion_matrix(y_test, predictions)
tn = cm[0][0]
fn = cm[0][1]
fp = cm[1][0]
tp = cm[1][1]

fp_svs = list(X_test.loc[(X_test['confirmed'] == 0) & (X_test['predictions'] == 1)].index)
fn_svs = list(X_test.loc[(X_test['confirmed'] == 1) & (X_test['predictions'] == 0)].index)

In [28]:
X_train

,size,confirmed,rep_LINE,rep_SINE,rep_LTR,rep_DNA,rep_Simple_repeat,rep_Satellite,rep_Low_complexity,rep_Retroposon,rep_snRNA,rep_tRNA,rep_srpRNA,rep_rRNA,rep_RC,rep_scRNA,rep_RNA,rep_VNTR,rep_STR,cpg_islands,centromeres,asmb_gaps,alt_haps,GC_content_left,GC_content_right,ill_cov_mean_I,ill_cov_mean_II,ill_cov_mean_III,ill_cov_mean_IV,ill_cov_std_I,ill_cov_std_II,ill_cov_std_III,ill_cov_std_IV,ill_isize_mean_I,ill_isize_mean_II,ill_isize_mean_III,ill_isize_mean_IV,ill_isize_std_I,ill_isize_std_II,ill_isize_std_III,ill_isize_std_IV,ill_mapq_mean_I,ill_mapq_mean_II,ill_mapq_mean_III,ill_mapq_mean_IV,ill_mapq_std_I,ill_mapq_std_II,ill_mapq_std_III,ill_mapq_std_IV,ill_clipreads_I,ill_clipreads_II,ill_clipreads_III,ill_clipreads_IV,ill_splitreads_I,ill_splitreads_II,ill_splitreads_III,ill_splitreads_IV,ill_disco_ff_I,ill_disco_ff_II,ill_disco_ff_III,ill_disco_ff_IV,ill_disco_rr_I,ill_disco_rr_II,ill_disco_rr_III,ill_disco_rr_IV
2616,31.0,0,0.0,1538.0,0.0,819.0,17753,135593.0,34660.0,20146.0,2733923.0,9738473,3275135.0,2590248,370442.0,9467444.0,6993480.0,25734.0,637.0,49615.0,92040323.0,137791.0,581055.0,29.8,30.6,-0.124,-1.426,-1.390,-0.143,-1.420,1.120,1.157,-1.187,0.113,0.054,0.080,0.074,0.640,0.611,0.591,0.504,0.043,0.019,0.018,0.016,-6.506,-0.648,-0.627,-0.582,0.000,0.418,0.431,0.000,0.100,0.185,0.190,0.102,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2617,238820255.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,52.0,0.111,0.010,-0.605,-0.303,-2.606,-2.918,-1.304,-2.467,15.708,0.051,13.718,13.355,20.079,1.053,19.138,18.961,-0.237,-0.024,-0.019,0.043,1.015,-0.454,-0.160,-6.506,0.013,0.000,0.022,0.018,0.000,0.000,0.000,0.000,0.054,0.044,0.024,0.019,0.000,0.000,0.024,0.037
2618,32708836.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,59271920.0,0.0,0.0,65.0,87.0,0.847,0.015,0.696,9.760,0.528,0.147,3.916,11.418,14.554,13.997,13.631,13.418,18.907,18.398,18.795,18.651,-3.346,-3.808,-2.448,-3.019,0.190,-0.043,0.737,0.519,0.293,0.331,0.645,0.548,0.305,0.404,0.593,0.730,0.114,0.142,0.036,0.027,0.120,0.156,0.479,0.512
2619,621.0,0,803.0,185.0,4808.0,4740.0,419,336209.0,5091.0,220762.0,2532717.0,9537267,3073929.0,2389042,169236.0,9266238.0,6792274.0,7433.0,5654.0,33237.0,91839117.0,338407.0,379849.0,36.8,24.6,-0.017,-0.105,0.113,0.050,-3.351,-1.846,-1.423,-3.108,-0.007,-0.041,0.086,0.124,0.967,0.478,1.212,1.287,0.043,0.040,0.043,0.043,-6.506,-2.912,-6.506,-6.506,0.015,0.029,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2620,32184180.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,59271917.0,0.0,0.0,71.2,88.8,-1.552,-2.444,1.181,9.890,-3.193,-1.316,4.270,11.314,11.759,11.897,13.607,13.419,16.697,16.762,18.756,18.651,-0.024,-0.152,-2.537,-3.033,0.471,0.849,0.715,0.513,0.045,0.500,0.633,0.546,0.000,0.050,0.614,0.731,0.045,0.050,0.035,0.027,0.000,0.000,0.486,0.512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26929,1775.0,1,0.0,211.0,469.0,1748.0,5927,4901353.0,85.0,51365.0,169881.0,1830143,2538989.0,99064,41608.0,2916807.0,406042.0,93483.0,5295.0,42738.0,88712926.0,4903652.0,71028386.0,31.6,36.4,0.337,-0.336,-1.239,-0.506,-1.928,-0.761,-3.139,-0.959,0.692,0.673,-0.077,0.586,3.370,3.359,0.594,3.268,0.091,0.089,0.098,0.098,-3.051,-2.940,-7.116,-7.116,0.000,0.189,0.237,0.000,0.120,0.143,0.000,0.116,0.012,0.014,0.000,0.000,0.012,0.014,0.000,0.000
26930,5196.0,1,0.0,0.0,0.0,3365.0,2335,4484768.0,22289.0,35754.0,583045.0,1413558,2122404.0,512228,369362.0,2500222.0,819206.0,6948.0,0.0,141239.0,89126090.0,4487067.0,71441550.0,59.0,52.2,-0.596,-0.982,-1.125,-0.810,-1.496,-2.340,-2.207,-2.008,1.252,0.728,0.564,1.100,4.753,4.312,3.854,4.516,0.061,0.052,0.062,0.059,-0.848,-0.696,-1.458,-1.507,0.000,0.000,0.028,0.049

In [26]:
"""
fp_svs = set()
fn_svs = set()
for chrom in CHROMS:
    print(chrom)
    curr_fp_svs, curr_fn_svs = training(df, chrom)
    fp_svs.update(curr_fp_svs)
    fn_svs.update(curr_fn_svs)
    print(len(fp_svs), len(fn_svs))
"""

'\nfp_svs = set()\nfn_svs = set()\nfor chrom in CHROMS:\n    print(chrom)\n    curr_fp_svs, curr_fn_svs = training(df, chrom)\n    fp_svs.update(curr_fp_svs)\n    fn_svs.update(curr_fn_svs)\n    print(len(fp_svs), len(fn_svs))\n'